# 🔬 C2C Teacher Mask Üretimi - DICOMNET Verileri

Bu notebook DICOMNET lokal verilerinden Comp2Comp kullanarak VAT, SAT, psoas teacher maskelerini üretir.

**Gereksinimler:**
- Colab Pro+ (GPU T4 veya A100)
- DICOMNET verileri Google Drive'da
- ~2-3 saat (45 vaka için)

**Çıktı:**
- `C2C_teachers_local/` klasörü
- Her vaka için: VAT, SAT, psoas segmentation masks (.nii.gz)
- Manifest JSON dosyası

## 1️⃣ Ortam Hazırlığı

In [ ]:
# GPU kontrolü
!nvidia-smi

In [ ]:
# Google Drive mount
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Çalışma dizini ayarla
import os
os.chdir('/content')

# DICOMNET yolunu ayarla (Google Drive'daki konumunuza göre)
DICOMNET_PATH = '/content/drive/MyDrive/DICOMNET'
OUTPUT_PATH = '/content/drive/MyDrive/C2C_teachers_local'

print(f"📂 DICOM Root: {DICOMNET_PATH}")
print(f"📂 Output: {OUTPUT_PATH}")

# Klasör var mı kontrol
if not os.path.exists(DICOMNET_PATH):
    print(f"❌ DICOMNET klasörü bulunamadı: {DICOMNET_PATH}")
    print("👉 Lütfen DICOMNET_PATH değişkenini güncelleyin")
else:
    case_count = len([d for d in os.listdir(DICOMNET_PATH) if os.path.isdir(os.path.join(DICOMNET_PATH, d)) and not d.startswith('.')])
    print(f"✅ {case_count} vaka bulundu")

## 2️⃣ Comp2Comp Kurulumu

In [ ]:
# Comp2Comp repo klonla
!git clone https://github.com/StanfordMIMI/Comp2Comp.git
%cd Comp2Comp

In [ ]:
# Colab ortamındaki mevcut torch/torchvision ile uyumlu kurulum
print("🔍 Mevcut torch sürümü:")
try:
    import torch, torchvision
    print(f"  torch: {torch.__version__}")
    print(f"  torchvision: {torchvision.__version__}")
except:
    print("  torch yüklü değil, kurulum yapılacak")
    %pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121

# setup.py'deki sabit torch 2.5.1 gereksinimini esnekleştir
import os
setup_path = '/content/Comp2Comp/setup.py'
if os.path.exists(setup_path):
    with open(setup_path, 'r') as f:
        setup_content = f.read()
    
    # torch/torchvision versiyonlarını esnet
    setup_content = setup_content.replace('torch==2.5.1', 'torch>=2.0.0')
    setup_content = setup_content.replace('torchvision==0.20.1', 'torchvision>=0.15.0')
    
    with open(setup_path, 'w') as f:
        f.write(setup_content)
    print("✅ setup.py torch versiyonu esnekleştirildi")

# Temel bağımlılıklar
%pip install -q pydicom moviepy h5py tabulate tqdm silx yacs pandas opencv-python huggingface_hub wget blosc2 acvl-utils==0.2 dicom2nifti<2.6 xgboost

# dosma (numpy uyumluluk kontrollü)
%pip install -q 'numpy<2.0' dosma

# pycocotools (isteğe bağlı)
try:
    %pip install -q pycocotools
    print("✅ pycocotools kuruldu")
except Exception as e:
    print(f"⚠️ pycocotools atlandı: {e}")

# comp2comp: editable install ile PYTHONPATH'e ekle
import sys
sys.path.insert(0, '/content/Comp2Comp')
print("✅ comp2comp modülü PYTHONPATH'e eklendi")

# Modül erişimini test et
try:
    import comp2comp
    print(f"✅ comp2comp {comp2comp.__version__} yüklü")
except Exception as e:
    print(f"❌ comp2comp import hatası: {e}")

print("\n✅ Kurulum tamamlandı")

# Kritik modül doğrulama
import importlib
for m in ["pydicom","dosma","torch","cv2","huggingface_hub","tqdm"]:
    try:
        importlib.import_module(m)
        print(f"✅ {m}")
    except Exception as e:
        print(f"❌ {m}: {e}")

In [ ]:
# numpy uyumluluk düzeltmesi
metrics_file = '/content/Comp2Comp/comp2comp/metrics/metrics.py'

with open(metrics_file, 'r') as f:
    content = f.read()

content = content.replace('.astype(np.bool)', '.astype(bool)')

with open(metrics_file, 'w') as f:
    f.write(content)

print("✅ numpy uyumluluk düzeltmesi yapıldı")

In [ ]:
# C2C binary düzeltmeleri
c2c_bin = '/content/Comp2Comp/bin/C2C'

with open(c2c_bin, 'r') as f:
    content = f.read()

# muscle_adipose_tissue wrapper ekle (yoksa)
if 'def MuscleAdiposeTissueWrapper' not in content:
    insert_pos = content.find('def SpinePipelineBuilder')
    wrapper = '''def MuscleAdiposeTissueWrapper(path, args):
    pipeline = InferencePipeline([io.DicomFinder(path), MuscleAdiposeTissuePipelineBuilder(args)])
    return pipeline\n\n'''
    content = content[:insert_pos] + wrapper + content[insert_pos:]

# main() düzelt
if 'args.pipeline == "muscle_adipose_tissue"' not in content:
    content = content.replace(
        'def main():\n    args = argument_parser().parse_args()\n    if args.pipeline == "spine_muscle_adipose_tissue":',
        'def main():\n    args = argument_parser().parse_args()\n    if args.pipeline == "muscle_adipose_tissue":\n        process_3d(args, MuscleAdiposeTissueWrapper)\n    elif args.pipeline == "spine_muscle_adipose_tissue":'
    )

content = content.replace('format(args.action)', 'format(args.pipeline)')

with open(c2c_bin, 'w') as f:
    f.write(content)

print("✅ C2C binary düzeltildi")

In [ ]:
# C2C test
!python /content/Comp2Comp/bin/C2C --help

## 3️⃣ Teacher Mask Üretim Fonksiyonları

In [ ]:
import os
import sys
import json
import subprocess
from pathlib import Path
from tqdm.auto import tqdm
from datetime import datetime

# comp2comp modülünü PYTHONPATH'e ekle (kurulum hücresinde yapılmışsa tekrar gerek yok, ama güvenlik için)
if '/content/Comp2Comp' not in sys.path:
    sys.path.insert(0, '/content/Comp2Comp')

def find_dicom_files(case_dir):
    case_path = Path(case_dir)
    dicom_files = list(case_path.rglob('*.dcm'))
    if len(dicom_files) == 0:
        for f in case_path.rglob('*'):
            if f.is_file() and not f.name.startswith('.'):
                try:
                    import pydicom
                    pydicom.dcmread(str(f), stop_before_pixels=True)
                    dicom_files.append(f)
                except:
                    pass
    return dicom_files

def run_c2c_inference(case_dir, output_dir, timeout=3600):
    c2c_bin = '/content/Comp2Comp/bin/C2C'
    
    # PYTHONPATH'i subprocess'e aktar
    env = os.environ.copy()
    env['PYTHONPATH'] = '/content/Comp2Comp:' + env.get('PYTHONPATH', '')
    
    cmd = [
        'python', c2c_bin, 'muscle_adipose_tissue',
        '--input_path', str(case_dir),
        '--output_path', str(output_dir),
        '--save_segmentations',
        '--muscle_fat_model', 'abCT_v0.0.1'
    ]
    
    try:
        result = subprocess.run(cmd, capture_output=True, timeout=timeout, text=True, env=env)
        if result.returncode != 0:
            print(f"    ⚠️  Exit code: {result.returncode}")
            if result.stderr:
                print(f"    stderr: {result.stderr[:500]}")
            return False
        
        output_path = Path(output_dir)
        timestamp_dirs = [d for d in output_path.iterdir() if d.is_dir()]
        if len(timestamp_dirs) == 0:
            return False
        
        latest_dir = sorted(timestamp_dirs)[-1]
        case_name = Path(case_dir).name
        case_output = latest_dir / case_name
        
        if not case_output.exists():
            return False
        
        seg_files = list(case_output.rglob('*.nii.gz'))
        if len(seg_files) == 0:
            return False
        
        print(f"    ✅ {len(seg_files)} mask oluşturuldu")
        return True
    except subprocess.TimeoutExpired:
        print(f"    ❌ Timeout ({timeout}s)")
        return False
    except Exception as e:
        print(f"    ❌ Hata: {e}")
        return False

print("✅ Fonksiyonlar hazır")

## 4️⃣ Teacher Mask Üretimi

In [ ]:
LIMIT = None
TIMEOUT = 3600

output_root = Path(OUTPUT_PATH)
output_root.mkdir(exist_ok=True, parents=True)

dicom_root = Path(DICOMNET_PATH)
case_dirs = [d for d in dicom_root.iterdir() if d.is_dir() and not d.name.startswith('.')]
if LIMIT:
    case_dirs = case_dirs[:LIMIT]

print(f"🔬 C2C Teacher Mask Üretimi")
print(f"✅ {len(case_dirs)} vaka işlenecek\n")

success = 0
error = 0
skip = 0
start = datetime.now()
manifest = {'created_at': start.isoformat(), 'cases': {}}

for case_dir in tqdm(case_dirs, desc="İşleniyor"):
    case_id = case_dir.name
    out_dir = output_root / case_id
    
    if out_dir.exists():
        existing = list(out_dir.rglob('*.nii.gz'))
        if len(existing) > 0:
            skip += 1
            manifest['cases'][case_id] = {'status': 'skipped'}
            continue
    
    dicoms = find_dicom_files(case_dir)
    if len(dicoms) == 0:
        error += 1
        manifest['cases'][case_id] = {'status': 'error', 'reason': 'no_dicom'}
        continue
    
    out_dir.mkdir(exist_ok=True, parents=True)
    if run_c2c_inference(case_dir, out_dir, TIMEOUT):
        success += 1
        manifest['cases'][case_id] = {'status': 'success', 'dicom_files': len(dicoms)}
    else:
        error += 1
        manifest['cases'][case_id] = {'status': 'error', 'reason': 'inference_failed'}

end = datetime.now()
manifest['completed_at'] = end.isoformat()
manifest['duration_seconds'] = (end - start).total_seconds()
manifest['success'] = success
manifest['error'] = error
manifest['skipped'] = skip

with open(output_root / 'c2c_manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)

print(f"\n✅ Tamamlandı!")
print(f"Başarılı: {success}, Hata: {error}, Atlandı: {skip}")
print(f"Süre: {manifest['duration_seconds']/60:.1f} dk")

## 5️⃣ Sonuçları Doğrula

In [ ]:
with open(Path(OUTPUT_PATH) / 'c2c_manifest.json', 'r') as f:
    manifest = json.load(f)

print("📊 Özet:")
print(f"Başarılı: {manifest['success']}")
print(f"Hata: {manifest['error']}")
print(f"Atlandı: {manifest['skipped']}")
print(f"Süre: {manifest['duration_seconds']/60:.1f} dk")

errors = [(k, v) for k, v in manifest['cases'].items() if v['status'] == 'error']
if errors:
    print(f"\n❌ Hatalı vakalar: {len(errors)}")
    for case_id, info in errors[:5]:
        print(f"  {case_id}: {info.get('reason')}")

In [ ]:
masks = list(Path(OUTPUT_PATH).rglob('*.nii.gz'))
print(f"\n📁 {len(masks)} mask dosyası oluşturuldu")
for m in masks[:5]:
    print(f"  {m.name}: {m.stat().st_size/1024/1024:.2f} MB")